# this script is used to convert TBOT and VBOT into TAUX and TAUY

In [1]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import os
import dask
from dask_jobqueue import PBSCluster
from dask.distributed import Client

1. Define constant variables

In [2]:
SHR_CONST_G = 9.80616
shr_const_karman = 0.4
alz = np.log(64./10.)  # need to confirm if zbot = 64
rho_air = 1.2 #kg/m3
umin = 0.5
#momentum_factor = 10.


2. Define functions

In [3]:
def cdn(umps):
#    if umps == 0:
 #       return 0.
 #   else:
        return  0.0027 / umps + 0.000142 + 0.0000764 * umps

In [15]:
def cdn1(umps):
    if umps.any() > 12.5:
        return shr_const_karman**2 / (np.log(10000.0/(0.085*umps-0.58)))**2
    else:
        return 0.0027 / umps + 0.000142 + 0.0000764 * umps
  


In [16]:
def qsat(t):
    return 640380.0 / np.exp(5107.4/t)

In [17]:
def psimhu(xd):
    return np.log((1.0+xd*(2.0+xd))*(1.0+xd*xd)/8.0) - 2.0*np.arctan(xd) + 1.571

## Start calculations


1. read in variables

In [20]:
cluster = PBSCluster(processes=20,
                     memory="6GB",
                     project='P93300014',
                     queue='regular',
                     resource_spec='select=1:ncpus=36:mem=109G',
                     walltime='1:00:00')

from dask.distributed import Client
client = Client(cluster)
print(cluster.job_script())
cluster.scale(2)


#!/usr/bin/env bash

#PBS -N dask-worker
#PBS -q regular
#PBS -A P93300014
#PBS -l select=1:ncpus=36:mem=109G
#PBS -l walltime=1:00:00

/glade/work/huili7/miniconda3/envs/geocat_env/bin/python -m distributed.cli.dask_worker tcp://10.148.10.15:34653 --nthreads 0 --nprocs 20 --memory-limit 300.00MB --name name --nanny --death-timeout 60 --interface ib0



/glade/work/huili7/miniconda3/envs/geocat_env/lib/python3.7/site-packages/distributed/dashboard/core.py:79: UserWarning: 
Port 8787 is already in use. 
Perhaps you already have a cluster running?
Hosting the diagnostics dashboard on a random port instead.
  warnings.warn("\n" + msg)


In [9]:
cluster.worker_spec

{0: {'cls': dask_jobqueue.pbs.PBSJob,
  'options': {'processes': 20,
   'memory': '6GB',
   'project': 'P93300014',
   'queue': 'regular',
   'resource_spec': 'select=1:ncpus=36:mem=109G',
   'walltime': '1:00:00',
   'config_name': 'pbs',
   'interface': 'ib0',
   'protocol': 'tcp://',
   'security': None},
  'group': ['-0',
   '-1',
   '-2',
   '-3',
   '-4',
   '-5',
   '-6',
   '-7',
   '-8',
   '-9',
   '-10',
   '-11',
   '-12',
   '-13',
   '-14',
   '-15',
   '-16',
   '-17',
   '-18',
   '-19']}}

In [21]:
def get_tau(file):
    print(file)
    data = xr.open_dataset('/glade/work/huili7/tc_wind_forcing/tc_files/year70-79/'+file)
    ubot = data['UBOT']
    vbot = data['VBOT']
    u10 = data['U10']
    time = data['time']
    date = data['date']
    dyear = data['dyear']
    nlat = u10.sizes['nlat']
    nlon = u10.sizes['nlon']

    taux = xr.DataArray(ubot.data, dims=['nlat', 'nlon'])
    taux.name = 'TAUX'
    taux.attrs['units'] = 'N/m^2'
    taux.attrs['long_name'] = 'Windstress in grid-x direction'
    taux.attrs['remap'] = 'remapped via ESMF_regrid_with_weights: Higher-order Patch'
    taux.values = 0. * taux.values

    tauy = xr.DataArray(ubot.data, dims=['nlat', 'nlon'])
    tauy.name = 'TAUY'
    tauy.attrs['units'] = 'N/m^2'
    tauy.attrs['long_name'] = 'Windstress in grid-y direction'
    tauy.attrs['remap'] = 'remapped via ESMF_regrid_with_weights: Higher-order Patch'
    tauy.values = 0. * tauy.values


    taux.encoding = {'original_shape': (384, 320),'dtype': np.dtype('float32'),
     'missing_value': 9.96921e+36,
     '_FillValue': 9.96921e+36}
    tauy.encoding = taux.encoding

# rdn * u10n = rd * vmag
# ustar = rd * vmag = rdn * u10n

    rdn = np.sqrt(cdn1(u10))
    rdn = np.where(rdn==np.inf, 0., rdn)

    ustar = rdn * u10
    tau = rho_air * ustar * ustar
    vmag = np.sqrt(ubot**2 + vbot**2)
    vmag = np.where(vmag>umin, vmag, umin)
    taux.values = tau * (ubot / vmag)
    tauy.values = tau * (vbot / vmag)

#    taux.values = taux.values * momentum_factor   # convert N/m2 to dyne/cm2
#    tauy.values = tauy.values * momentum_factor  !! no longer needed. This is doen in the model code on the fly. 

    date_str = str(date.values[0])
    print(date_str)
    yyy = date_str[0:2]    # if we are dealing with  under 100 years [0:2]
    doy = str(dyear.values[0] + 1000)
    hhh = date_str[6:8]    # 6:8 if dealing with under 100 years

    filetime = '00'+yyy+'.'+doy[1:4]+'.'+hhh   # yyy is two digits when deaeling with under 100
    print(filetime)
    filename = '/glade/work/huili7/tc_wind_forcing/tc_tau_files/year70-99-mcd/b.e13.B1850C5CN.ne120_g16.005.tau.'+filetime

    if os.path.isfile(filename+'.nc'):
        print ("File exist")
        os.remove(filename+'.nc')
    
    taux.to_netcdf(filename+'.nc')
    tauy.to_netcdf(filename+'.nc',mode='a')
    time.to_netcdf(filename+'.nc',mode='a')
    del(data)

### the next section is for embarrasingly parallelization

In [22]:
file_list = os.listdir('/glade/work/huili7/tc_wind_forcing/tc_files/year70-79')
file_list.sort()
nfiles = len(file_list)


%time

for i in range(0, nfiles,20):
    lazy_results = []
    
    for j in range(i,i+20):
        print(file_list[j])
        lazy_result = dask.delayed(get_tau)(file_list[j])
        lazy_results.append(lazy_result)
    dask.compute(*lazy_results)
    lazy_results.clear()

CPU times: user 2 µs, sys: 1 µs, total: 3 µs
Wall time: 5.25 µs
b.e13.B1850C5CN.ne120_g16.005.UVBOT.70010100.nc
b.e13.B1850C5CN.ne120_g16.005.UVBOT.70010106.nc
b.e13.B1850C5CN.ne120_g16.005.UVBOT.70010112.nc
b.e13.B1850C5CN.ne120_g16.005.UVBOT.70010118.nc
b.e13.B1850C5CN.ne120_g16.005.UVBOT.70010200.nc
b.e13.B1850C5CN.ne120_g16.005.UVBOT.70010206.nc
b.e13.B1850C5CN.ne120_g16.005.UVBOT.70010212.nc
b.e13.B1850C5CN.ne120_g16.005.UVBOT.70010218.nc
b.e13.B1850C5CN.ne120_g16.005.UVBOT.70010300.nc
b.e13.B1850C5CN.ne120_g16.005.UVBOT.70010306.nc
b.e13.B1850C5CN.ne120_g16.005.UVBOT.70010312.nc
b.e13.B1850C5CN.ne120_g16.005.UVBOT.70010318.nc
b.e13.B1850C5CN.ne120_g16.005.UVBOT.70010400.nc
b.e13.B1850C5CN.ne120_g16.005.UVBOT.70010406.nc
b.e13.B1850C5CN.ne120_g16.005.UVBOT.70010412.nc
b.e13.B1850C5CN.ne120_g16.005.UVBOT.70010418.nc
b.e13.B1850C5CN.ne120_g16.005.UVBOT.70010500.nc
b.e13.B1850C5CN.ne120_g16.005.UVBOT.70010506.nc
b.e13.B1850C5CN.ne120_g16.005.UVBOT.70010512.nc
b.e13.B1850C5CN.ne120_g1

KeyboardInterrupt: 

In [12]:
u10

NameError: name 'u10' is not defined